# D07_S01 Refugee Camp for 6 Homes
Design the system for a  120V, AC distribution system for 24 households. You have access to the same 50mx50m square to deploy the PV and batteries and single phase inverters.

The requirement is to supply electricity for an ADMD of 300W, and that the loads are assumed to be a constant power load, with pf =0.95 . The national regulator states that all AC systems must be within 10% of the nominal voltage. After testing of the existing lines, it is found that they have a ratio of X/R = 1. (Use R = 0.37 Ω)

Extend your Gauss Seidel code to model the system to determine the voltage at each node.
1) Determine Pbase Vbase  and Zbase .
2) Use R from the DC case, and convert to PU. Calc X. 
3) Do GS using PU values
4) Revert to actual values if you want to.


In [1]:
# Import numpy
import numpy as np


## Set up the constants


In [2]:
# Inputs
P = 300 # Phome ADMD
pf = 0.95 # power factor

VSlack = 130 + 0j# should be a complex number

VBase = 120
SBase = 1000

ZBase = VBase**2 / SBase # ohms law

R = 0.37 # line resistance in Ohms

XR = 1 # X/R ratio 

# Calculations
V0 = VSlack / VBase # per unit voltage at slack bus

Rpu = R / ZBase  # per unit line resistance
Xpu = Rpu * XR  # per unit line reactance

Y = 1 / (Rpu + 1j * Xpu)  # line admittance

Ph = P / SBase  # power of the home in per unit

Qh = Ph * np.tan(np.arccos(pf))   # reactive power of the home





In [3]:
# Print out values
print("S")
print(f" S_h = {Ph*SBase:.4f} + j {Qh*SBase:.4f} ")
print(f" S_h_pu = {Ph:.4f} + j {Qh:.4f} ")
print(f" Y = {Y:.4f} ZBase = = {ZBase:.4f}  ")


S
 S_h = 300.0000 + j 98.6052 
 S_h_pu = 0.3000 + j 0.0986 
 Y = 19.4595-19.4595j ZBase = = 14.4000  


## This block puts it all together
We have 6 unknown voltages

In [4]:
# Initialise the 6 bus voltages.
V1_k = 1+0j
V2_k = 1+0j
V3_k = 1+0j
V4_k = 1+0j
V5_k = 1+0j
V6_k = 1+0j

# Set up the error criterion
eps = 1e-6 # TODO


# set max iterations
max_iter = 10000 # TODO

# Set the error so that the while loop will start
e = 1.0

# Set up a counter to count the number of iterations
count = 0

Sh_conj = Ph - 1j*Qh   # conj of home load (pu)

# instigate the while loop
while e > eps and count < max_iter:
    count += 1

    # calculate the new voltages (load injection = -(Ph + jQh))
    # Home 1
    V1_k1 = (-Sh_conj / np.conj(V1_k) + Y*(V0    + V2_k)) / (2*Y)
    # Home 2
    V2_k1 = (-Sh_conj / np.conj(V2_k) + Y*(V1_k1 + V3_k)) / (2*Y)
    # Home 3
    V3_k1 = (-Sh_conj / np.conj(V3_k) + Y*(V2_k1 + V4_k)) / (2*Y)
    # Home 4
    V4_k1 = (-Sh_conj / np.conj(V4_k) + Y*(V3_k1 + V5_k)) / (2*Y)
    # Home 5
    V5_k1 = (-Sh_conj / np.conj(V5_k) + Y*(V4_k1 + V6_k)) / (2*Y)
    # Home 6 (end of feeder: only one line connected)
    V6_k1 = (-Sh_conj / np.conj(V6_k) + Y*V5_k1) / Y

    # calculate total error
    e = (abs(V1_k1 - V1_k) + abs(V2_k1 - V2_k) + abs(V3_k1 - V3_k)
         + abs(V4_k1 - V4_k) + abs(V5_k1 - V5_k) + abs(V6_k1 - V6_k))
    
  #  print(f"{count:4} V0 = {V0:.1f} 1 = {V1_k1:.3f} 2 = {V2_k1:.3f} 3 = {V3_k1:.3f} 4 = {V4_k1:.3f} 5 = {V5_k1:.3f} 6 = {np.abs(V6_k1):.3f} err6 = {err3:9.4g}")
    print(f"{count:4} V1 = {np.abs(V1_k1):.4f} V6 = {np.abs(V6_k1):.4f}  err = {e:9.4g}")
    
    # update the voltages for the next iteration
    V1_k = V1_k1
    V2_k = V2_k1    
    V3_k = V3_k1
    V4_k = V4_k1
    V5_k = V5_k1
    V6_k = V6_k1

# handle non-convergence
    if (count >= max_iter):
      print(f"WARNING DID NOT CONVERGE IN {count} ITERATIONS ")
    #else:
      #print(f"Converged in {count} iterations")

   1 V1 = 1.0365 V6 = 0.9825  err =   0.09082
   2 V1 = 1.0433 V6 = 0.9717  err =   0.04814
   3 V1 = 1.0454 V6 = 0.9630  err =   0.03899
   4 V1 = 1.0459 V6 = 0.9557  err =   0.03575
   5 V1 = 1.0457 V6 = 0.9490  err =   0.03427
   6 V1 = 1.0449 V6 = 0.9430  err =    0.0332
   7 V1 = 1.0437 V6 = 0.9373  err =   0.03186
   8 V1 = 1.0423 V6 = 0.9320  err =   0.03034
   9 V1 = 1.0410 V6 = 0.9270  err =   0.02875
  10 V1 = 1.0396 V6 = 0.9222  err =   0.02718
  11 V1 = 1.0383 V6 = 0.9177  err =   0.02566
  12 V1 = 1.0371 V6 = 0.9135  err =   0.02421
  13 V1 = 1.0359 V6 = 0.9094  err =   0.02285
  14 V1 = 1.0348 V6 = 0.9056  err =   0.02156
  15 V1 = 1.0337 V6 = 0.9020  err =   0.02034
  16 V1 = 1.0326 V6 = 0.8985  err =    0.0192
  17 V1 = 1.0317 V6 = 0.8952  err =   0.01813
  18 V1 = 1.0307 V6 = 0.8921  err =   0.01712
  19 V1 = 1.0299 V6 = 0.8891  err =   0.01617
  20 V1 = 1.0290 V6 = 0.8863  err =   0.01528
  21 V1 = 1.0282 V6 = 0.8836  err =   0.01444
  22 V1 = 1.0274 V6 = 0.8810  err 

## Print out the final voltages

In [5]:
for i, v in enumerate([V1_k, V2_k, V3_k, V4_k, V5_k, V6_k], start=1):
    print(f"V{i} = {np.abs(v):.4f} ∠ {np.angle(v, deg=True):.2f}°")

V1 = 1.0126 ∠ -1.62°
V2 = 0.9528 ∠ -3.16°
V3 = 0.9044 ∠ -4.53°
V4 = 0.8677 ∠ -5.67°
V5 = 0.8431 ∠ -6.48°
V6 = 0.8307 ∠ -6.90°


## Determine all the currents and power flows

In [6]:
# Slack bus P+jQ Use equation (4) Not -jQ 
I01 = Y * (V0 - V1_k)
S0 = V0 * np.conj(I01)# TODO

print(f"S0 = {S0 * SBase:.4f} (one feeder)") 

S0 = 2103.3529+894.9910j (one feeder)


In [7]:
# currents
I12 = Y * (V1_k - V2_k);  I21 = -I12
I23 = Y * (V2_k - V3_k);  I32 = -I23
I34 = Y * (V3_k - V4_k);  I43 = -I34
I45 = Y * (V4_k - V5_k);  I54 = -I45
I56 = Y * (V5_k - V6_k);  I65 = -I56


In [8]:
# power
S01 = V0   * np.conj(I01);  S10 = V1_k * np.conj(-I01)
S12 = V1_k * np.conj(I12);  S21 = V2_k * np.conj(I21)
S23 = V2_k * np.conj(I23);  S32 = V3_k * np.conj(I32)
S34 = V3_k * np.conj(I34);  S43 = V4_k * np.conj(I43)
S45 = V4_k * np.conj(I45);  S54 = V5_k * np.conj(I54)
S56 = V5_k * np.conj(I56);  S65 = V6_k * np.conj(I65)

lines = [("0-1", I01, S01, S10), ("1-2", I12, S12, S21), ("2-3", I23, S23, S32),
         ("3-4", I34, S34, S43), ("4-5", I45, S45, S54), ("5-6", I56, S56, S65)]
Sloss = 0
print(f"{'line':>5} {'|I| (A)':>8} {'S_send (VA)':>22} {'S_recv (VA)':>22} {'loss (VA)':>18}")
for name, I, Ss, Sr in lines:
    loss = Ss + Sr
    Sloss += loss
    print(f"{name:>5} {abs(I)*IBase:8.3f} {Ss*SBase:22.2f} {Sr*SBase:22.2f} {loss*SBase:18.2f}")
print(f"Feeder losses = {Sloss*SBase:.2f} VA  ({Sloss.real/S0.real*100:.1f}% of P sent)")

 line  |I| (A)            S_send (VA)            S_recv (VA)          loss (VA)


NameError: name 'IBase' is not defined

 ## Max ADMD
If the supply voltage is increased to a nominal voltage of 230V, what is the maximum ADMD that can be supplied using the original DC lines, whilst still meeting the Voltage requirements.
